# MiniMax H3 (DaSiWa cMMH3 V19) — MANUEL (ComfyUI arayüzünde elle) — Colab

ComfyUI'yi tünelle açar, grafiği **sen elle sürersin**. Bu bir *deneme*: MiniMax H3 ailesinin queen-editor'e alınmaya değip değmediği burada görülecek (v5 yol haritası, madde 213).

**Input:** bir fotoğraf + prompt (ComfyUI UI'da) · **Output:** sesli video — H3 videoyu ve sesi birlikte üretir

Sıra:
1. **CONFIG** — tek anahtar: REF2VA da inecek mi
2. **Ortak Yardımcılar** — log + fail-loud run + model doğrulama
3. **ComfyUI + Manager + custom node'lar** (6)
4. **Modeller** — ~41,8 GB, hepsi HuggingFace, gated değil
5. **Başlat + cloudflared tünel** → UI linki

> Drive kullanılmaz; modeller her oturumda kaynaktan iner (Colab geçici diski).
>
> **Runtime → Change runtime type → A100** — 21 GB'lık model + 15 GB'lık metin kodlayıcı.
>
> **Run all** → en alttaki linke gir → `workflow.json`'u kendi bilgisayarından sürükle-bırak → grafik **I2VA** modunda açılır → Director'e fotoğraf + prompt → **Queue Prompt**.
>
> ⚠️ **`nvfp4` / `fp8` dosyalarını seçme.** A100 Ampere: fp4 Blackwell'de, fp8 Ada/Hopper'da yerel. Bu defter bilerek `int8_convrot` (UNET) ve `int4_convrot` (CLIP) indiriyor.
>
> Dosyaların tam listesi ve neden bu sürümler: **`indirilecekler.md`** · modlar, ayarlar, tuzaklar: **`instructions.md`**

## 1) CONFIG

Doldurulacak bir şey yok — tek karar REF2VA. Gerisine dokunma → **Run all**.

In [ ]:
# === CONFIG ===
# Gated indirme yok: MiniMax H3'ün ağırlıkları HuggingFace'te açık duruyor. Civitai cookie'si
# yalnız DaSiWa'nın kendi turbo checkpoint'i için gerekir; o ikinci koşuya bırakıldı
# (indirilecekler.md § 3).

# REF2VA kendi UNET'ini istiyor (+21 GB). Öteki dört mod -- T2VA, I2VA, FL2VA, L2VA -- FL2VA'nın
# UNET'ini paylaşıyor, ve deneme I2VA ile başlıyor. O yüzden varsayılan kapalı.
WITH_REF2VA = False

COMFY_PORT = 8188

import subprocess

def _sh(cmd):
    return subprocess.run(cmd, capture_output=True, text=True).stdout.strip()

print("=== GPU ===")
print(_sh(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
           "--format=csv,noheader"]) or "(nvidia-smi cevap vermedi)")
print("=== Disk ===")
print(_sh(["df", "-h", "/content"]))

# 21 GB UNET + 15 GB metin kodlayıcı = 36 GB. Sırayla yüklenirse 40 GB'lık bir A100'e sığar, aynı
# anda sığmaz. Bu bir uyarı, hata değil: küçük kartta da Chunking açılarak denenebilir.
_vram = _sh(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"]).splitlines()
if _vram and _vram[0].strip().isdigit() and int(_vram[0]) < 38000:
    print(f"\n⚠️  {_vram[0]} MiB VRAM — model 21 GB, metin kodlayıcı 15 GB.")
    print("    Sığmazsa Director'deki '⚙️ Chunking' (MiniMax H3 Chunk FeedForward) anahtarını aç.")

## 2) Ortak Yardımcılar

`log` + fail-loud `run` + model doğrulama. Doğrulama **ağa soru sormaz**: safetensors header'ındaki `data_offsets` beklenen boyutu verir. HEAD/`Content-Length` kullanılmaz — HF'in Xet CDN'i imzalı URL'de HEAD'e 403 döner ve o hata gövdesi "dosya boyutu" sanılır.

In [ ]:
# === Shared helpers — log + fail-loud run + model validation ===
# Used by section 3 (custom nodes) and section 4 (model download); defined once (DRY).
import os, json, time, struct, subprocess

def log(msg, level="INFO"):
    icons = {"INFO": "ℹ️ ", "OK": "✅", "WARN": "⚠️ ", "ERR": "❌"}
    print(f"{icons.get(level, '·')} [{time.strftime('%H:%M:%S')}] {msg}")

def human(b):
    """Bytes -> human-readable size (e.g. 1.5GB)."""
    for u in ["B", "KB", "MB", "GB"]:
        if b < 1024:
            return f"{b:.1f}{u}"
        b /= 1024
    return f"{b:.1f}TB"

def head_text(path, limit=4000):
    """First bytes of a file as raw text — the response body, printed as-is, not interpreted."""
    if not os.path.exists(path):
        return "(dosya yok)"
    size = os.path.getsize(path)
    with open(path, "rb") as f:
        text = f.read(limit).decode("utf-8", errors="replace")
    return text + (f"\n… (+{human(size - limit)})" if size > limit else "")

def run(cmd, label, cwd=None, timeout=3600):
    """Run a command; non-zero exit or timeout -> RuntimeError with the command's own stderr.

    The single gate for download failures: the downloader exits non-zero on an HTTP error, on a
    transfer that ends before the announced length, and on a full disk.
    """
    try:
        r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd,
                           capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"{label}: timeout ({timeout}s)")
    if r.returncode != 0:
        tail = "\n".join((r.stderr or r.stdout or "").strip().splitlines()[-5:])
        raise RuntimeError(f"{label}: exit {r.returncode}\n{tail}")
    return r.stdout

def check_safetensors(path):
    """State of a model file -> ("ok" | "partial" | "invalid", msg).

    The expected total size is computed from the file itself: a safetensors file is
    [8-byte LE header length][header JSON][tensor data], and the header's data_offsets say where
    the tensor data ends. No Content-Length, no HEAD request (HF's Xet CDN answers HEAD with 403
    while serving the GET fine, so a HEAD-based size check reads the error body as the size).

    ok      -> header parses and the file is exactly as long as its header says
    partial -> valid prefix, shorter than expected: safe to resume
    invalid -> empty / error page / longer than expected: garbage, stop
    """
    if not os.path.exists(path):
        return "invalid", "missing"
    size = os.path.getsize(path)
    if size < 8:
        return "invalid", f"too small ({human(size)})"

    with open(path, "rb") as f:
        header_len = struct.unpack("<Q", f.read(8))[0]
        if not (0 < header_len < 200_000_000):
            return "invalid", f"bad header length ({header_len})"
        if 8 + header_len > size:
            return "partial", f"header incomplete ({human(size)})"
        try:
            header = json.loads(f.read(header_len).decode("utf-8"))
        except (UnicodeDecodeError, json.JSONDecodeError) as e:
            return "invalid", f"header parse failed ({type(e).__name__}, {human(size)})"

    ends = [v["data_offsets"][1] for k, v in header.items()
            if k != "__metadata__" and isinstance(v, dict) and "data_offsets" in v]
    if not ends:                        # metadata-only header: nothing to measure against
        return "ok", f"{human(size)}, no tensor offsets"

    expected = 8 + header_len + max(ends)
    if size == expected:
        return "ok", f"{human(size)}, {len(ends)} tensors"
    if size < expected:
        return "partial", f"{size:,} / {expected:,} bytes"
    return "invalid", f"too long: {size:,} / {expected:,} bytes"

print("✓ Ortak yardımcılar hazır (log, run, human, head_text, check_safetensors)")

## 3) ComfyUI + Manager + Custom Node'lar (6)

Liste grafiğin kendi **"📋 Features & Requirements"** notundan alındı. Manager o listede yok; ayrıca kuruluyor çünkü `MiniMaxH3Director`, `MiniMaxH3Cache`, `MiniMaxChunkFeedForward` gibi node'ların hangi paketten geldiği **bilinmiyor** — grafiğin notu "native H3 backend" diyor, yani güncel ComfyUI'nin kendisinden bekleniyor. Eksik çıkarsa UI'da **Manager → Install Missing Custom Nodes**.

Biri başarısız olursa hücre `RuntimeError` ile durur (fail-loud).

In [ ]:
%cd /content

# === System deps + ComfyUI ===
# ffmpeg: DaSiWa EnhancedVideoCombine onunla encode ediyor (grafiğin Requirements notu).
!apt-get install -y ffmpeg aria2 > /dev/null 2>&1
![ -d ComfyUI ] || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!git pull -q
!pip install -q -r requirements.txt

# === Custom nodes (fail-loud: clone or pip failure -> RuntimeError) ===
import os
%cd /content/ComfyUI/custom_nodes

# (folder, repo) — trailing comment = what the node provides to this graph
CUSTOM_NODES = [
    ("ComfyUI-Manager",              "https://github.com/ltdrdata/ComfyUI-Manager.git"),                 # install missing nodes from the UI
    ("rgthree-comfy",                "https://github.com/rgthree/rgthree-comfy.git"),                    # Label, Reroute
    ("ComfyUI-KJNodes",              "https://github.com/kijai/ComfyUI-KJNodes.git"),                    # ModelPreviewOverrideKJ
    ("ComfyUI-GGUF",                 "https://github.com/city96/ComfyUI-GGUF.git"),                      # UnetLoaderGGUF
    ("ComfyUI-DaSiWa-Nodes",         "https://github.com/darksidewalker/ComfyUI-DaSiWa-Nodes.git"),      # Director, Watermark, EnhancedVideoCombine, NodeStatusSwitch
    ("Comfyui-MMH3-UltimateUpscale", "https://github.com/bbaudio-2025/Comfyui-MMH3-UltimateUpscale.git"),# MMH3UltimateUpscale + split params
]

for name, url in CUSTOM_NODES:
    if os.path.exists(name) and os.listdir(name):
        log(f"{name}: zaten var")
        continue
    log(f"{name}: cloning...")
    run(["git", "clone", "--depth", "1", url, name], f"clone {name}", timeout=180)
    if not os.listdir(name):                 # clone reported success but folder is empty
        raise RuntimeError(f"{name}: klon sonrası klasör boş")
    req = f"/content/ComfyUI/custom_nodes/{name}/requirements.txt"
    if os.path.exists(req):
        run(f"pip install -q -r {req}", f"pip install {name}", timeout=600)  # install node deps

log(f"{len(CUSTOM_NODES)} custom node hazır", "OK")

## 4) Modeller — ~41,8 GB, hepsi HuggingFace

Grafiğin **"⚙️ Settings"** tablosunun seçtiği dosyalar. Gated indirme yok, cookie yok.

| Klasör | Dosya | Boyut |
|---|---|---|
| `diffusion_models` | `minimax_h3_fl2va_pruned_int8_convrot` | 21 GB |
| `text_encoders` | `qwen3vl_32b_minimax_h3_int4_convrot` | 15 GB |
| `vae` | `minimax_h3_video_vae_fp16` | 5,21 GB |
| `vae` | `minimax_h3_audio_vae_fp32` | 605 MB |
| `vae_approx` | `taeh3` | küçük |

**FL2VA UNET dört modu birden koşturuyor** — T2VA, I2VA, FL2VA, L2VA. Ayrı UNET yalnız REF2VA istiyor; `WITH_REF2VA` onun için.

Upscale, RIFE, watermark ve latent upscale modelleri **inmiyor**: grafikte altı anahtarın altısı da kapalı geliyor, kapalı grup tümden bypass.

Herhangi bir dosya bozuk/eksik inerse hücre `RuntimeError` ile durur; bozuk dosya **silinmez**, inceleme için diskte kalır.

In [ ]:
import os, glob

COMFY = "/content/ComfyUI"
for d in ["diffusion_models", "text_encoders", "vae", "vae_approx"]:
    os.makedirs(f"{COMFY}/models/{d}", exist_ok=True)

def fetch(url, target_dir, filename, label):
    """Download + validate a model; anything invalid stops the run (fail-loud, nothing deleted).

    aria2c first -- 16 connections, and these are 21 GB files. HF's Xet CDN answers parallel byte
    ranges with 403 on some assets; then, and only then, curl runs it on one connection. The first
    error is carried into the second's message rather than replaced by it: a retry that hides why
    the first attempt failed is how a dead URL gets read as a slow network.

    Downloads land in <target>.part and are renamed only once check_safetensors says "ok", so
    ComfyUI never sees a half-written file under the real model name.
    """
    target = os.path.join(target_dir, filename)
    part = target + ".part"

    if os.path.exists(target):
        state, msg = check_safetensors(target)
        if state == "ok":
            log(f"{label}: zaten var ({msg})")
            return
        raise RuntimeError(f"{label}: {state} — {msg}\n{target}\n--- file head ---\n{head_text(target)}")

    state, msg = check_safetensors(part) if os.path.exists(part) else ("missing", "yok")
    if state == "invalid":
        # Resuming onto garbage would append good bytes to it and hide the problem.
        raise RuntimeError(f"{label}: .part {state} — {msg}\n{part}\n--- file head ---\n{head_text(part)}")

    if state == "ok":
        log(f"{label}: .part zaten tam ({msg}) — indirilmiyor")
    else:
        log(f"{label}: iniyor ({msg})")
        aria = ["aria2c", "-x", "16", "-s", "16", "-k", "1M", "--continue=true",
                "--console-log-level=warn", "--auto-file-renaming=false",
                "--allow-overwrite=true", "-d", target_dir, "-o", os.path.basename(part), url]
        try:
            run(aria, f"{label} (aria2c)", timeout=7200)
        except RuntimeError as first:
            log(f"{label}: aria2c düştü — tek bağlantıyla curl deneniyor", "WARN")
            curl = ["curl", "-L", "-C", "-", "--fail-with-body", "--max-time", "7200", "-o", part, url]
            try:
                run(curl, f"{label} (curl)", timeout=7500)
            except RuntimeError as second:
                raise RuntimeError(f"{first}\n\n{second}\n{url}\n"
                                   f"--- file head ---\n{head_text(part)}") from None

    state, msg = check_safetensors(part)
    if state != "ok":
        raise RuntimeError(f"{label}: {state} — {msg}\n{part}\n{url}\n"
                           f"--- file head ---\n{head_text(part)}")
    os.replace(part, target)
    log(f"{label}: indirildi ve doğrulandı ({msg})", "OK")

# === The files the graph's Settings panel selects ===
# int8 (UNET) and int4 (CLIP) on purpose: the nvfp4 and fp8_scaled builds of the same weights are
# native only on Blackwell and Ada/Hopper, and Colab's big card is an A100 (Ampere).
HF = "https://huggingface.co"
CO = f"{HF}/Comfy-Org/MiniMax-H3/resolve/main"

MODELS = [
    # (url, target_dir, filename, label)
    (f"{CO}/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors",
     f"{COMFY}/models/diffusion_models", "minimax_h3_fl2va_pruned_int8_convrot.safetensors",
     "UNET FL2VA int8 (21 GB)"),
    # The text encoder is Qwen3-VL 32B. Comfy-Org's int8 build is 27.1 GB; this int4 one is what
    # the graph itself names, and it is 12 GB smaller.
    (f"{HF}/Abiray/MiniMax-H3-GGUF/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_int4_convrot.safetensors",
     f"{COMFY}/models/text_encoders", "qwen3vl_32b_minimax_h3_int4_convrot.safetensors",
     "CLIP Qwen3-VL 32B int4 (15 GB)"),
    (f"{CO}/vae/minimax_h3_video_vae_fp16.safetensors",
     f"{COMFY}/models/vae", "minimax_h3_video_vae_fp16.safetensors", "Video VAE fp16 (5.2 GB)"),
    # Not optional: the graph's own note says the audio VAE is required for REF2VA, and every mode
    # here produces sound alongside the picture.
    (f"{CO}/vae/minimax_h3_audio_vae_fp32.safetensors",
     f"{COMFY}/models/vae", "minimax_h3_audio_vae_fp32.safetensors", "Audio VAE fp32 (605 MB)"),
    (f"{HF}/Kijai/MiniMax-H3-TAE/resolve/main/vae_approx/taeh3.safetensors",
     f"{COMFY}/models/vae_approx", "taeh3.safetensors", "TAE önizleme"),
]

if WITH_REF2VA:
    MODELS.append((f"{CO}/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",
                   f"{COMFY}/models/diffusion_models",
                   "minimax_h3_ref2va_pruned_int8_convrot.safetensors",
                   "UNET REF2VA int8 (21 GB)"))

for url, d, fn, label in MODELS:
    fetch(url, d, fn, label)

# === Summary (reaching here means everything downloaded + validated) ===
for d in ["diffusion_models", "text_encoders", "vae", "vae_approx"]:
    print(f"\n📂 {d}/")
    for f in sorted(glob.glob(f"{COMFY}/models/{d}/*.safetensors")):
        print(f"   {human(os.path.getsize(f))}  {os.path.basename(f)}")
log("Tüm modeller indirildi ve doğrulandı", "OK")

## 5) Başlat + cloudflared Tünel

ComfyUI arka planda başlar (90 sn içinde `/system_stats` cevap vermezse fail-loud), tünel linki basılır, sonra hücre **bilerek açık kalır** — biterse Colab runtime'ı idle sayıp tüneli öldürür.

In [ ]:
import subprocess, time, urllib.request, re, os

if not os.path.isfile("/content/cloudflared"):
    run(["wget", "-q", "-O", "/content/cloudflared",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], "cloudflared")
    run(["chmod", "+x", "/content/cloudflared"], "chmod cloudflared")

# Re-run safety: kill the previous ComfyUI + tunnel before starting new ones
subprocess.run(["pkill", "-f", "main.py"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(2)

# --enable-manager: on current ComfyUI the Manager is OFF without this flag. The workflow is loaded
# by hand and some of its H3 nodes are unaccounted for, so the Manager has to stay reachable.
logf = open("/content/comfyui.log", "w")
subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT), "--enable-manager"],
                 cwd="/content/ComfyUI", stdout=logf, stderr=subprocess.STDOUT)
ok = False
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=2)
        ok = True
        break
    except Exception:
        pass
if not ok:
    print("".join(open("/content/comfyui.log").readlines()[-30:]))
    raise RuntimeError("❌ ComfyUI 90 sn içinde başlamadı — yukarıdaki log'a bak")
log(f"ComfyUI ayakta ({(i+1)*2}s)", "OK")

# cloudflared output goes to a file, not a pipe: an unread pipe fills up and blocks the process.
tunlog = "/content/cloudflared.log"
subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{COMFY_PORT}"],
                 stdout=open(tunlog, "w"), stderr=subprocess.STDOUT)
link = None
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[-\w.]+trycloudflare\.com", open(tunlog).read()) if os.path.exists(tunlog) else None
    if m:
        link = m.group(0)
        break
if not link:
    print(open(tunlog).read()[-1000:] if os.path.exists(tunlog) else "(cloudflared log yok)")
    raise RuntimeError("❌ cloudflared linki 30 sn içinde alınamadı")

print(f"\n🔗 ComfyUI linki: {link}\n")
print("⬆️  Linke gir → workflow.json'u kendi bilgisayarından sürükle-bırak")
print("    Grafik I2VA modunda açılır: Director'e bir fotoğraf ver, prompt'u yaz, Queue Prompt (Ctrl+Enter)")
print("    Settings panelinde seçili olmalı:")
print("      UNET      minimax_h3_fl2va_pruned_int8_convrot")
print("      CLIP      qwen3vl_32b_minimax_h3_int4_convrot")
print("      Video VAE minimax_h3_video_vae_fp16    Audio VAE  minimax_h3_audio_vae_fp32")
print("      Steps 25, sampler res_multistep — bu checkpoint turbo DEĞİL (turbo 4-8 adım, Civitai'de)")
print("⚠️  'MiniMax H3 Cache'i ilk denemede açma — grafiğin kendi uyarısı: ghost/morph yapabilir")
print("⚠️  VRAM yetmezse: Director → '⚙️ Chunking' (MiniMax H3 Chunk FeedForward)")
print("    Eksik node olursa: Manager → Install Missing Custom Nodes → Restart")
print("    Beğenirsen: Workflow → Export (API) — madde 213 queen-editor'e onu koyacak\n")

# === Keep the cell OPEN (critical) ===
# ComfyUI + the tunnel run in the background. If this cell ENDS, Colab can call the runtime idle
# and disconnect -> ComfyUI + link die. Streaming the log keeps the cell in the foreground and
# shows generation progress when Run is pressed in the UI.
# To stop: interrupt this cell (■) or Runtime -> Disconnect.
print("📡 ComfyUI çalışıyor — BU HÜCREYİ KAPATMA. Canlı log:\n")
try:
    subprocess.run(["tail", "-n", "+1", "-f", "/content/comfyui.log"])
except KeyboardInterrupt:
    log("Hücre durduruldu — ComfyUI hâlâ arka planda çalışıyor (yeni link için bu hücreyi tekrar çalıştır).", "WARN")